# Tensors & Einsum — Complete Reference
### CS221 Practice Notebook

This notebook covers every einsum operation from easy to advanced.
Read the comments carefully — they explain WHY, not just WHAT.

In [1]:
import numpy as np
from einops import einsum

---
## PART 1 — Understanding Tensor Shapes

Before einsum, you need to understand shapes.
`.shape` tells you the size of each axis.

In [2]:
# Order 0 — Scalar (no axes, just a number)
scalar = np.array(42)
print('Scalar shape:', scalar.shape)   # () means no axes at all
print('Scalar value:', scalar)
print()

Scalar shape: ()
Scalar value: 42



In [3]:
# Order 1 — Vector (one axis)
vector = np.array([1, 2, 3])
print('Vector shape:', vector.shape)   # (3,) means 1 axis with 3 elements
print('Vector value:', vector)
print()

Vector shape: (3,)
Vector value: [1 2 3]



In [4]:
# Order 2 — Matrix (two axes: rows and columns)
matrix = np.array([[1, 2, 3],
                   [4, 5, 6]])
print('Matrix shape:', matrix.shape)   # (2, 3) means 2 rows, 3 columns
print('Matrix value:')
print(matrix)
print()

Matrix shape: (2, 3)
Matrix value:
[[1 2 3]
 [4 5 6]]



In [5]:
# Order 3 — Tensor (three axes: like stacked spreadsheets)
tensor3d = np.array([[[1, 2], [3, 4]],
                     [[5, 6], [7, 8]]])
print('3D Tensor shape:', tensor3d.shape)  # (2, 2, 2) means 2 sheets, 2 rows, 2 cols
print('3D Tensor value:')
print(tensor3d)

3D Tensor shape: (2, 2, 2)
3D Tensor value:
[[[1 2]
  [3 4]]

 [[5 6]
  [7 8]]]


---
## PART 2 — Einsum on Vectors (Order 1)

### The ONE Rule of Einsum:
> For every combination of input positions, **multiply** the values,
> then **put the result** in the matching output position.
> If an axis is **missing from the output** → sum it up and collapse it.

### Reading the string:
```
"input_axes -> output_axes"
```
- Left of `->` = input axes
- Right of `->` = output axes  
- Missing from right = that axis gets summed away
- Empty right side = scalar output

In [6]:
# Our vector for all examples below
x = np.array([1, 2, 3])
print('x =', x, '| shape:', x.shape)

x = [1 2 3] | shape: (3,)


In [7]:
# ── OPERATION 1: Identity ──────────────────────────────────────────
# String: 'i -> i'
# Reading: axis i goes in, axis i comes out
# Effect: just copy the vector, nothing changes
# Output shape: same as input → vector (3,)

y = einsum(x, 'i -> i')
print('Identity  |  i -> i  |  output:', y)   # [1, 2, 3]

Identity  |  i -> i  |  output: [1 2 3]


In [8]:
# ── OPERATION 2: Sum All Elements ─────────────────────────────────
# String: 'i ->'
# Reading: axis i goes in, NOTHING comes out
# Effect: i disappears from output → sum everything → one number
# Output shape: scalar () — no axes at all

y = einsum(x, 'i ->')
print('Sum all   |  i ->    |  output:', y)    # 1+2+3 = 6

Sum all   |  i ->    |  output: 6


In [9]:
# ── OPERATION 3: Element-wise Product ─────────────────────────────
# String: 'i, i -> i'
# Reading: two vectors, both with axis i, output keeps axis i
# Effect: multiply matching positions, keep each result
# Output shape: same vector (3,)
# KEY: same 'i' in both inputs means MATCHING positions only

y = einsum(x, x, 'i, i -> i')
print('Elementwise product  |  i,i -> i  |  output:', y)  # [1,4,9]

Elementwise product  |  i,i -> i  |  output: [1 4 9]


In [10]:
# ── OPERATION 4: Dot Product ───────────────────────────────────────
# String: 'i, i ->'
# Reading: two vectors with axis i, output is EMPTY
# Effect: multiply matching positions, then sum everything
# Output shape: scalar ()
# DIFFERENCE from elementwise: right side is empty → collapses to one number

y = einsum(x, x, 'i, i ->')
print('Dot product  |  i,i ->  |  output:', y)  # 1*1+2*2+3*3 = 14

Dot product  |  i,i ->  |  output: 14


In [11]:
# ── OPERATION 5: Outer Product ────────────────────────────────────
# String: 'i, j -> i j'
# Reading: first vector uses axis i, second uses axis j
# Effect: try EVERY combination of i and j, multiply them
# Output shape: matrix (3, 3) — because i has 3 and j has 3
# KEY: i and j are DIFFERENT axes → every pair, not just matching
# Think: multiplication table

y = einsum(x, x, 'i, j -> i j')
print('Outer product  |  i,j -> ij  |  output:')
print(y)
# [[1*1, 1*2, 1*3],
#  [2*1, 2*2, 2*3],
#  [3*1, 3*2, 3*3]]

Outer product  |  i,j -> ij  |  output:
[[1 2 3]
 [2 4 6]
 [3 6 9]]


In [12]:
# ── SPOT THE DIFFERENCE ───────────────────────────────────────────
# This is the most important comparison to understand

elementwise = einsum(x, x, 'i, i -> i')   # same i → matching positions only
outer       = einsum(x, x, 'i, j -> i j') # i AND j → every combination

print('Elementwise (i,i->i):', elementwise)  # vector [1,4,9]
print('Outer       (i,j->ij):')
print(outer)                                  # 3x3 matrix

Elementwise (i,i->i): [1 4 9]
Outer       (i,j->ij):
[[1 2 3]
 [2 4 6]
 [3 6 9]]


---
## PART 3 — Einsum on Matrices (Order 2)

Now we have TWO axes: `i` for rows, `j` for columns.
The same one rule applies — just more axes to track.

In [13]:
# Our matrix for all examples below
# Shape (2, 3) = 2 rows, 3 columns
M = np.array([[1, 2, 3],
              [4, 5, 6]])
print('M =')
print(M)
print('shape:', M.shape)  # (2, 3)

M =
[[1 2 3]
 [4 5 6]]
shape: (2, 3)


In [14]:
# ── OPERATION 1: Identity ──────────────────────────────────────────
# String: 'i j -> i j'
# Reading: both axes go in, both axes come out
# Effect: copy the matrix exactly
# Output shape: same matrix (2, 3)

y = einsum(M, 'i j -> i j')
print('Identity  |  ij -> ij  |  output:')
print(y)

Identity  |  ij -> ij  |  output:
[[1 2 3]
 [4 5 6]]


In [15]:
# ── OPERATION 2: Sum ALL Elements ─────────────────────────────────
# String: 'i j ->'
# Reading: both axes go in, NOTHING comes out
# Effect: both i and j disappear → everything collapses to one number
# Output shape: scalar ()

y = einsum(M, 'i j ->')
print('Sum all  |  ij ->  |  output:', y)  # 1+2+3+4+5+6 = 21

Sum all  |  ij ->  |  output: 21


In [16]:
# ── OPERATION 3: Row Sums ──────────────────────────────────────────
# String: 'i j -> i'
# Reading: both axes in, only i comes out
# Effect: j disappears → sum across columns for each row
# Output shape: vector (2,) — one value per row
# Think: 'collapse the columns, keep the rows'

y = einsum(M, 'i j -> i')
print('Row sums  |  ij -> i  |  output:', y)  # [1+2+3, 4+5+6] = [6, 15]

Row sums  |  ij -> i  |  output: [ 6 15]


In [17]:
# ── OPERATION 4: Column Sums ───────────────────────────────────────
# String: 'i j -> j'
# Reading: both axes in, only j comes out
# Effect: i disappears → sum across rows for each column
# Output shape: vector (3,) — one value per column
# Think: 'collapse the rows, keep the columns'

y = einsum(M, 'i j -> j')
print('Col sums  |  ij -> j  |  output:', y)  # [1+4, 2+5, 3+6] = [5, 7, 9]

Col sums  |  ij -> j  |  output: [5 7 9]


In [18]:
# ── OPERATION 5: Transpose ────────────────────────────────────────
# String: 'i j -> j i'
# Reading: flip the order of axes in the output
# Effect: rows become columns, columns become rows
# Output shape: (3, 2) — flipped from (2, 3)
# No transposes needed — just swap the axis names!

y = einsum(M, 'i j -> j i')
print('Transpose  |  ij -> ji  |  output:')
print(y)  # shape (3, 2)

Transpose  |  ij -> ji  |  output:
[[1 4]
 [2 5]
 [3 6]]


In [19]:
# ── OPERATION 6: Element-wise Product (Matrix) ────────────────────
# String: 'i j, i j -> i j'
# Reading: two matrices, same axes, output keeps same axes
# Effect: multiply each position by the matching position
# Output shape: same (2, 3)

y = einsum(M, M, 'i j, i j -> i j')
print('Elementwise product  |  ij,ij -> ij  |  output:')
print(y)  # each element squared

Elementwise product  |  ij,ij -> ij  |  output:
[[ 1  4  9]
 [16 25 36]]


In [20]:
# ── OPERATION 7: Matrix-Vector Product ────────────────────────────
# String: 'i j, j -> i'
# Reading: matrix has axes i,j — vector has axis j — output keeps i
# Effect: j is SHARED → multiply matching j positions, sum them → one value per row
# Output shape: vector (2,) — one value per row of the matrix
# This is the core operation in linear predictors: M @ v

v = np.array([1, 0, 2])   # shape (3,)
y = einsum(M, v, 'i j, j -> i')
print('Matrix-vector  |  ij,j -> i  |  output:', y)
# Row 0: 1*1 + 2*0 + 3*2 = 7
# Row 1: 4*1 + 5*0 + 6*2 = 16

Matrix-vector  |  ij,j -> i  |  output: [ 7 16]


In [21]:
# ── OPERATION 8: Matrix-Matrix Product ───────────────────────────
# String: 'i k, k j -> i j'
# Reading: A has axes i,k — B has axes k,j — output has i,j
# Effect: k is SHARED → sum over k → classic matrix multiplication
# Output shape: (rows of A, cols of B)
# KEY: k disappears → it gets summed away

A = np.array([[1, 2],
              [3, 4]])   # shape (2, 2)

B = np.array([[2, 0],
              [1, 3]])   # shape (2, 2)

y = einsum(A, B, 'i k, k j -> i j')
print('Matrix-matrix  |  ik,kj -> ij  |  output:')
print(y)
# Same as A @ B

Matrix-matrix  |  ik,kj -> ij  |  output:
[[ 4  6]
 [10 12]]


In [22]:
# ── OPERATION 9: Matrix-Matrix (Different Sizes) ──────────────────
# A is (2,3), B is (3,4) → result is (2,4)
# The shared axis k must match in size: A has 3 cols, B has 3 rows ✅

A = np.array([[1, 2, 3],
              [4, 5, 6]])        # shape (2, 3)

B = np.array([[1, 0, 1, 0],
              [0, 1, 0, 1],
              [1, 1, 0, 0]])     # shape (3, 4)

y = einsum(A, B, 'i k, k j -> i j')
print('Rect matmul  |  ik,kj -> ij  |  output shape:', y.shape)
print(y)

Rect matmul  |  ik,kj -> ij  |  output shape: (2, 4)
[[ 4  5  1  2]
 [10 11  4  5]]


---
## PART 4 — Common Mistakes & How to Avoid Them

These are the mistakes that trip everyone up.

In [23]:
# ── MISTAKE 1: Using wrong number of axes ─────────────────────────
# A matrix has 2 axes (i and j). You MUST name both.
# Using 'i -> i' on a matrix FAILS because you only named 1 axis.

M = np.array([[1, 2, 3], [4, 5, 6]])

try:
    y = einsum(M, 'i -> i')   # WRONG for matrix — only names 1 axis
except ValueError as e:
    print('ERROR (expected):', 'Too few axes named for this tensor')

# CORRECT for matrix:
y = einsum(M, 'i j -> i j')  # name BOTH axes
print('Correct matrix identity:', y.shape)

ERROR (expected): Too few axes named for this tensor
Correct matrix identity: (2, 3)


In [24]:
# ── MISTAKE 2: Scalar vs Vector output confusion ──────────────────
# 'i ->'   → scalar, a single number, shape ()
# 'i -> i' → vector, a list, shape (3,)

x = np.array([1, 2, 3])

scalar_out = einsum(x, 'i ->')
vector_out = einsum(x, 'i -> i')

print('Scalar output:', scalar_out, '| shape:', scalar_out.shape)  # 6, shape ()
print('Vector output:', vector_out, '| shape:', vector_out.shape)  # [1,2,3], shape (3,)

Scalar output: 6 | shape: ()
Vector output: [1 2 3] | shape: (3,)


In [25]:
# ── MISTAKE 3: i,i vs i,j — the most important difference ─────────
# 'i, i -> i' = matching positions only (element-wise)
# 'i, j -> ij' = every combination (outer product)

x = np.array([1, 2, 3])

elementwise = einsum(x, x, 'i, i -> i')   # shape (3,)
outer       = einsum(x, x, 'i, j -> i j') # shape (3, 3)

print('Same axis (i,i->i)   — elementwise:', elementwise)
print('Diff axis (i,j->ij) — outer product:')
print(outer)

Same axis (i,i->i)   — elementwise: [1 4 9]
Diff axis (i,j->ij) — outer product:
[[1 2 3]
 [2 4 6]
 [3 6 9]]


---
## PART 5 — Quick Reference Cheat Sheet

| Operation | String | Output Shape | What it does |
|---|---|---|---|
| Vector identity | `i -> i` | (n,) | Copy the vector |
| Vector sum | `i ->` | scalar | Add all elements |
| Elementwise product | `i, i -> i` | (n,) | Multiply matching positions |
| Dot product | `i, i ->` | scalar | Multiply then sum all |
| Outer product | `i, j -> i j` | (n, n) | Every combination |
| Matrix identity | `i j -> i j` | (m, n) | Copy the matrix |
| Sum all | `i j ->` | scalar | Add every element |
| Row sums | `i j -> i` | (m,) | Sum each row |
| Column sums | `i j -> j` | (n,) | Sum each column |
| Transpose | `i j -> j i` | (n, m) | Flip rows and columns |
| Matrix-vector | `i j, j -> i` | (m,) | Transform a vector |
| Matrix-matrix | `i k, k j -> i j` | (m, n) | Standard matmul |

---
## PART 6 — Practice Problems

Try these yourself before running the cell below each one.

In [26]:
# PRACTICE 1:
# Given x = [2, 4, 6], write einsum to get the sum of squares
# (2² + 4² + 6² = 4 + 16 + 36 = 56)
# Hint: you need two x's and output should be scalar

x = np.array([2, 4, 6])
# your answer here:
# result = einsum(x, x, '??? -> ???')
# print(result)

In [27]:
# PRACTICE 2:
# Given M below, get the sum of only the FIRST column
# (1 + 4 = 5)
# Hint: first get column sums, then index the result

M = np.array([[1, 2, 3],
              [4, 5, 6]])
# your answer here:
col_sums = einsum(M, 'i j -> j')  # sum across rows to get column sums
print(col_sums[0])

5


In [28]:
# PRACTICE 3:
# What is the shape of this output? Predict before running.
# A is (3, 4), B is (4, 2)

A = np.ones((3, 4))
B = np.ones((4, 2))
result = einsum(A, B, 'i k, k j -> i j')
print('Shape:', result.shape)  # what do you predict?

Shape: (3, 2)


In [29]:
# ANSWERS — only look after you've tried!

x = np.array([2, 4, 6])
M = np.array([[1, 2, 3], [4, 5, 6]])

# Practice 1: sum of squares
p1 = einsum(x, x, 'i, i ->')
print('Practice 1 — Sum of squares:', p1)  # 56

# Practice 2: first column sum
col_sums = einsum(M, 'i j -> j')
print('Practice 2 — First column sum:', col_sums[0])  # 5

# Practice 3: shape
A = np.ones((3, 4))
B = np.ones((4, 2))
p3 = einsum(A, B, 'i k, k j -> i j')
print('Practice 3 — Output shape:', p3.shape)  # (3, 2)

Practice 1 — Sum of squares: 56
Practice 2 — First column sum: 5
Practice 3 — Output shape: (3, 2)
